# **H2O validation and perplexity**

Cells copied unchanged (outputs kept) from `archive/OS_MAIN.ipynb`: original cell indices [0, 1, 31, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45].

In [1]:
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
import transformers

transformers.utils.logging.disable_progress_bar()

torch.cuda.set_per_process_memory_fraction(0.45)

# model_name = "gpt2"
model_name = "facebook/opt-6.7b"

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Model: facebook/opt-6.7b
Device: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from datasets import load_dataset

# WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Get long articles by filtering for substantial text
long_prompts = []
for item in dataset:
    text = item["text"].strip()
    tokens = tokenizer.encode(text)
    if len(tokens) > 200:  # only keep long articles
        long_prompts.append(text)
    if len(long_prompts) >= 32:
        break

print(f"Found {len(long_prompts)}  prompts")
for i, p in enumerate(long_prompts[:4]):
    tokens = tokenizer.encode(p)
    print(f"Prompt {i}: {len(tokens)} tokens | {p[:80]}...")

Found 32  prompts
Prompt 0: 217 tokens | He had a recurring role in 2003 on two episodes of The Bill , as character " Con...
Prompt 1: 205 tokens | In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill . The ...
Prompt 2: 246 tokens | Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont...
Prompt 3: 268 tokens | Du Fu 's work is notable above all for its range . Chinese critics traditionally...


In [ ]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

def compute_importance_scores(attentions):
    num_layers = len(attentions)
    num_heads = attentions[0].shape[1]
    num_queries = attentions[0].shape[2]
    num_keys = attentions[0].shape[3]

    scores = torch.zeros(num_keys).to(device)

    for n in range(num_layers):
        layer_attn = attentions[n][0]
        avg_across_heads = layer_attn.sum(dim=0) / num_heads
        avg_across_queries = avg_across_heads.sum(dim=0) / num_queries

        scores += avg_across_queries

    scores = scores / num_layers

    return scores


def classify_indices(top,middle,bottom,scores):
  # sorted scores must be passed here
  sorted_indices = scores.argsort(descending=True)
  top_length = int((len(sorted_indices)/100) * top)
  middle_length = int(((len(sorted_indices)/100)) * middle)
  bottom_lenght = int(((len(sorted_indices)/100)) * bottom)

  hot = sorted_indices[ :top_length]
  cold = sorted_indices[top_length:top_length+middle_length]
  evict = sorted_indices[top_length+middle_length: ]

  return hot,cold,evict

def split_cache(kv_cache, hot_indices, cold_indices):
    hot_layers = []
    cold_layers = []

    for k, v in kv_cache:
        h_idx = hot_indices.to(k.device)
        c_idx = cold_indices.to(k.device)

        hot_k = k[:, :, h_idx, :]
        hot_v = v[:, :, h_idx, :]
        hot_layers.append((hot_k, hot_v))

        cold_k = k[:, :, c_idx, :]
        cold_v = v[:, :, c_idx, :]
        cold_layers.append((cold_k, cold_v))

    return hot_layers, cold_layers

# **H2O VALIDATION**

In [ ]:
def validate_h2o_proper(prompt, max_new_tokens=200, budget_ratio=0.8, recent_ratio=0.5):
    """
    Proper H2O implementation:
    - Fixed cache budget = prompt_len * budget_ratio
    - Budget split: half for recent tokens, half for heavy hitters
    - Evict ONE lowest-scoring non-recent token when cache exceeds budget
    - Cumulative attention scores updated every step
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs.input_ids.shape[1]
    budget = int((prompt_len + max_new_tokens) * budget_ratio)
    recent_window = int(budget * recent_ratio)  # half budget for recent
    num_layers = model.config.num_hidden_layers

    # BASELINE
    with torch.no_grad():
        kv = DynamicCache()
        outputs = model(input_ids=inputs.input_ids, past_key_values=kv, use_cache=True)
        kv = outputs.past_key_values
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        baseline_tokens = [next_token.item()]
        for _ in range(max_new_tokens - 1):
            outputs = model(input_ids=next_token, past_key_values=kv, use_cache=True)
            kv = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
            baseline_tokens.append(next_token.item())

    # H2O
    with torch.no_grad():
        outputs = model(
            input_ids=inputs.input_ids,
            use_cache=True,
            output_attentions=True
        )
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        h2o_tokens = [next_token.item()]

        # Initialize cumulative scores from prefill
        seq_len = outputs.past_key_values.get_seq_length()
        cum_scores = torch.zeros(seq_len, device=device)
        for attn in outputs.attentions:
            cum_scores += attn[0].sum(dim=0).sum(dim=0)

        kv = outputs.past_key_values

        for step in range(max_new_tokens - 1):
            outputs = model(
                input_ids=next_token,
                past_key_values=kv,
                use_cache=True,
                output_attentions=True
            )
            kv = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
            h2o_tokens.append(next_token.item())

            # Update cumulative scores — new token starts at 0
            current_len = kv.get_seq_length()
            new_score = torch.zeros(1, device=device)
            cum_scores = torch.cat([cum_scores, new_score])

            # Add this step's attention scores
            for attn in outputs.attentions:
                step_scores = attn[0].sum(dim=0).sum(dim=0)
                if len(step_scores) == len(cum_scores):
                    cum_scores += step_scores

            # Evict if over budget — one at a time
            while current_len > budget:
                # Protect recent tokens — last recent_window positions
                evict_candidates = cum_scores[:-recent_window].clone()

                if len(evict_candidates) == 0:
                    break

                # Find lowest scoring non-recent token
                evict_pos = evict_candidates.argmin().item()

                # Remove from cache
                keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                keep_mask[evict_pos] = False
                keep_indices = torch.where(keep_mask)[0]

                kv_layers = unpack_kv_cache(kv)
                kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                kv = build_kv_cache(kept_layers)

                # Remove from scores
                cum_scores = cum_scores[keep_mask]
                current_len -= 1

            del outputs

    total = len(baseline_tokens)
    match = sum(a == b for a, b in zip(baseline_tokens, h2o_tokens))
    print(f"  Budget={budget} (ratio={budget_ratio}) | Recent window={recent_window}")
    print(f"  Match: {match}/{total} ({100*match/total:.1f}%)")
    print(f"  Baseline: {tokenizer.decode(baseline_tokens[:25])}")
    print(f"  H2O:      {tokenizer.decode(h2o_tokens[:25])}")

# Test with different budget ratios
for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    validate_h2o_proper(p, budget_ratio=0.9)
    validate_h2o_proper(p, budget_ratio=0.8)
    validate_h2o_proper(p, budget_ratio=0.7)


Prompt: Artificial intelligence has transformed the way we approach ...
  Budget=232 (ratio=0.9) | Recent window=116
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  Budget=206 (ratio=0.8) | Recent window=103
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems 

In [ ]:
for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    validate_h2o_proper(p, budget_ratio=0.5)
    validate_h2o_proper(p, budget_ratio=0.4)
    validate_h2o_proper(p, budget_ratio=0.3)


Prompt: Artificial intelligence has transformed the way we approach ...
  Budget=129 (ratio=0.5) | Recent window=64
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  Budget=103 (ratio=0.4) | Recent window=51
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in

In [ ]:
for p in LONG_PROMPTS[:4]:
    print(f"\nPrompt: {p[:60]}...")
    validate_h2o_proper(p, budget_ratio=0.2)


Prompt: Artificial intelligence has transformed the way we approach ...
  Budget=51 (ratio=0.2) | Recent window=25
  Match: 200/200 (100.0%)
  Baseline:  Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in
  H2O:       Artificial intelligence has transformed the way we approach complex problems in science and engineering Artificial intelligence has transformed the way we approach complex problems in

Prompt: The history of ancient Rome spans over a thousand years from...
  Budget=53 (ratio=0.2) | Recent window=26
  Match: 17/200 (8.5%)
  Baseline:  The history of ancient Rome spans over a thousand years from its founding as a small village The history of ancient Rome spans over a
  H2O:       The history of ancient Rome spans over a thousand years from its founding as a village The history of ancient Rome spans over a thousand

Prompt: Quantum computing l

In [ ]:
long_unique = """Large language models have revolutionized natural language processing by demonstrating remarkable capabilities in text generation, translation, summarization, and question answering. These models, built on the transformer architecture introduced by Vaswani et al in 2017, use self-attention mechanisms to capture long-range dependencies in text. The key innovation of transformers is the multi-head attention mechanism, which allows the model to attend to information from different representation subspaces at different positions. During inference, the model generates tokens autoregressively, producing one token at a time based on the entire previous context. This requires storing key and value tensors from all previous tokens in what is known as the KV cache. As sequence lengths grow, the KV cache becomes a significant memory bottleneck, consuming gigabytes of GPU memory for long contexts. Several approaches have been proposed to address this challenge, including quantization of cached values, eviction of less important entries, and offloading to CPU memory."""

validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.5)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.3)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.2)

  Budget=197 (ratio=0.5) | Recent window=98
  Match: 12/200 (6.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to address this problem, called the KV-cache-free model. The
  Budget=118 (ratio=0.3) | Recent window=59
  Match: 13/200 (6.5%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the memory burden of large language models. We use a novel memory
  Budget=78 (ratio=0.2) | Recent window=39
  Match: 14/200 (7.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the computation of the transformer to the GPU

In [ ]:
long_unique = """Large Language Models (LLMs), despite their recent impressive accomplishments,
are notably cost-prohibitive to deploy, particularly for applications involving long-
content generation, such as dialogue systems and story writing. Often, a large
amount of transient state information, referred to as the KV cache, is stored in GPU
memory in addition to model parameters, scaling linearly with the sequence length
and batch size. In this paper, we introduce a novel approach for implementing the
KV cache which significantly reduces its memory footprint. Our approach is based
on the noteworthy observation that a small portion of tokens contributes most of
the value when computing attention scores. We call these tokens Heavy Hitters
(H2). Through a comprehensive investigation, we find that (i) the emergence of H2
is natural and strongly correlates with the frequent co-occurrence of tokens in the
text, and (ii) removing them results in significant performance degradation. Based
on these insights, we propose Heavy Hitter Oracle (H2O), a KV cache eviction
policy that dynamically retains a balance of recent and H2 tokens. We formulate
the KV cache eviction as a dynamic submodular problem and prove (under mild
assumptions) a theoretical guarantee for our novel eviction algorithm which could
help guide future work. We validate the accuracy of our algorithm with OPT,
LLaMA, and GPT-NeoX across a wide range of tasks. Our implementation of
H2O with 20% heavy hitters improves the throughput over three leading inference
systems DeepSpeed Zero-Inference, Hugging Face Accelerate, and FlexGen by
up to 29×, 29×, and 3×on OPT-6.7B and OPT-30B. With the same batch size,
H2O can reduce the latency by up to 1.9×. The code is available at https:
//github.com/FMInference/H2O."""

validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.6)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.3)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.2)

  Budget=364 (ratio=0.6) | Recent window=182
  Match: 200/200 (100.0%)
  Baseline: </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  H2O:      </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  Budget=182 (ratio=0.3) | Recent window=91
  Match: 200/200 (100.0%)
  Baseline: </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  H2O:      </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  Budget=121 (ratio=0.2) | Recent window=60
  Match: 17/200 (8.5%)
  Baseline: </s>I'm not sure if this is a good thing or a bad thing.
It's a good thing.  
  H2O:      </s>I'm not sure if this is a good thing or a bad thing.
I'm not sure if this is a


In [ ]:
long_unique = """Large language models have revolutionized natural language processing by demonstrating remarkable capabilities in text generation, translation, summarization, and question answering. These models, built on the transformer architecture introduced by Vaswani et al in 2017, use self-attention mechanisms to capture long-range dependencies in text. The key innovation of transformers is the multi-head attention mechanism, which allows the model to attend to information from different representation subspaces at different positions. During inference, the model generates tokens autoregressively, producing one token at a time based on the entire previous context. This requires storing key and value tensors from all previous tokens in what is known as the KV cache. As sequence lengths grow, the KV cache becomes a significant memory bottleneck, consuming gigabytes of GPU memory for long contexts. Several approaches have been proposed to address this challenge, including quantization of cached values, eviction of less important entries, and offloading to CPU memory."""

validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.5)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.3)
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.2)

  Budget=197 (ratio=0.5) | Recent window=98
  Match: 12/200 (6.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to address this problem, called the KV-cache-free model. The
  Budget=118 (ratio=0.3) | Recent window=59
  Match: 13/200 (6.5%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the memory burden of large language models. We use a novel memory
  Budget=78 (ratio=0.2) | Recent window=39
  Match: 14/200 (7.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to offload the computation of the transformer to the GPU

In [ ]:
validate_h2o_proper(long_unique, max_new_tokens=200, budget_ratio=0.6)

  Budget=236 (ratio=0.6) | Recent window=118
  Match: 200/200 (100.0%)
  Baseline:  In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to
  H2O:       In this paper, we propose a novel approach to address the memory bottleneck of large language models. We propose a novel approach to


# **H2O Perplexity eval**

In [ ]:
def evaluate_perplexity_h2o(text, budget_ratio=1.0, recent_ratio=0.5, stride=512):
    """Compute perplexity with H2O eviction at given budget ratio. 1.0 = no eviction (baseline)."""
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    seq_len = input_ids.shape[1]

    nlls = []

    with torch.no_grad():
        # Process full sequence token by token
        kv = DynamicCache()
        cum_scores = None

        for i in range(0, seq_len - 1):
            token = input_ids[:, i:i+1]

            outputs = model(
                input_ids=token,
                past_key_values=kv,
                use_cache=True,
                output_attentions=True
            )
            kv = outputs.past_key_values

            # Compute NLL for next token prediction
            logits = outputs.logits[:, -1, :]
            target = input_ids[:, i + 1]
            nll = torch.nn.functional.cross_entropy(logits, target)
            nlls.append(nll.item())

            # Update cumulative scores
            current_len = kv.get_seq_length()
            step_scores = torch.zeros(current_len, device=device)
            for attn in outputs.attentions:
                scores = attn[0].sum(dim=0).sum(dim=0)
                if len(scores) == current_len:
                    step_scores += scores

            if cum_scores is None:
                cum_scores = step_scores
            else:
                new_score = torch.zeros(1, device=device)
                cum_scores = torch.cat([cum_scores, new_score])
                if len(cum_scores) == current_len:
                    cum_scores += step_scores

            # H2O eviction if over budget
            if budget_ratio < 1.0:
                budget = max(2, int(seq_len * budget_ratio))
                recent_window = int(budget * recent_ratio)

                while current_len > budget:
                    evict_candidates = cum_scores[:-recent_window].clone()
                    if len(evict_candidates) == 0:
                        break

                    evict_pos = evict_candidates.argmin().item()

                    keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                    keep_mask[evict_pos] = False
                    keep_indices = torch.where(keep_mask)[0]

                    kv_layers = unpack_kv_cache(kv)
                    kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                    kv = build_kv_cache(kept_layers)

                    cum_scores = cum_scores[keep_mask]
                    current_len -= 1

            del outputs

    avg_nll = sum(nlls) / len(nlls)
    perplexity = torch.exp(torch.tensor(avg_nll)).item()
    return perplexity, len(nlls)


# Get evaluation text — use a real paragraph
eval_text = """The transformer architecture was introduced in 2017 by Vaswani et al. in their landmark paper Attention Is All You Need. The architecture relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions entirely. Since its introduction, the transformer has become the foundation for nearly all modern large language models including GPT, BERT, T5, LLaMA, and many others. The key innovation was the multi-head self-attention mechanism which allows the model to attend to different positions in the input sequence simultaneously. This parallel attention computation made transformers significantly more efficient to train than recurrent neural networks which process tokens sequentially. The original transformer was designed for machine translation but researchers quickly discovered that the architecture could be adapted for virtually any natural language processing task. Decoder-only variants like GPT use causal attention masks to enable autoregressive generation where each token can only attend to previous tokens. During inference these models store key and value tensors from all previous tokens in what is known as the KV cache to avoid redundant computation. As sequence lengths and batch sizes grow the KV cache becomes a dominant consumer of GPU memory often exceeding the memory required for model parameters themselves. This has motivated extensive research into KV cache compression including quantization eviction policies and offloading strategies."""

print("Evaluating perplexity across different H2O budget ratios...")
print(f"Text length: {len(tokenizer.encode(eval_text))} tokens\n")

for ratio in [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2]:
    label = "Full (baseline)" if ratio == 1.0 else f"H2O {int(ratio*100)}% budget"
    ppl, n_tokens = evaluate_perplexity_h2o(eval_text, budget_ratio=ratio)
    print(f"  {label:<25} | Perplexity: {ppl:.2f} | Tokens: {n_tokens}")

Evaluating perplexity across different H2O budget ratios...
Text length: 260 tokens

  Full (baseline)           | Perplexity: 23.84 | Tokens: 259
  H2O 80% budget            | Perplexity: 23.85 | Tokens: 259
  H2O 60% budget            | Perplexity: 23.50 | Tokens: 259
  H2O 50% budget            | Perplexity: 23.51 | Tokens: 259
  H2O 40% budget            | Perplexity: 23.58 | Tokens: 259
  H2O 30% budget            | Perplexity: 24.48 | Tokens: 259
  H2O 20% budget            | Perplexity: 26.36 | Tokens: 259
